# Análisis del censo de viviendas 2021

Objetivo principal:
- Calcular el tamaño medio según el tipo de vivienda en Santa Cruz de Bezana

Carpeta de datos esperada:
- `data/raw/INE/CensoViviendas_2021.csv`


## 1. Setup: imports, rutas y configuración

Ajusta los nombres de archivo si has usado otra convención.

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

DATA_DIR = Path("../data/raw/INE")

CENSO_VIVIENDAS = DATA_DIR / "CensoViviendas_2021.csv"

## 2. Carga de datos

In [7]:
df = pd.read_csv(CENSO_VIVIENDAS,
                 sep=',',
                 index_col = 0)

print(f"Dataset con {df.shape[0]} registros y {df.shape[1]} columnas\n")
df.head()

Dataset con 2662371 registros y 11 columnas



,CPRO,CMUN,TIPO_VIV,TENEN_VIV,RT_imp,SUPERF,PLANTAS_sr,PLANTAS_br,TIPO_EDIF,ANOCO,TAM_HOG_5
ORDEN_V,,,,,,,,,,,
1,3,99,1,2.0,1.0,84.0,1.0,0.0,1.0,1980.0,1.0
2,24,991,1,2.0,0.0,94.0,1.0,0.0,1.0,1931.0,3.0
3,32,991,0,NaN,NaN,104.0,1.0,0.0,1.0,1980.0,NaN
4,25,993,1,2.0,0.0,152.0,1.0,0.0,1.0,2005.0,3.0
5,30,30,1,3.0,1.0,69.0,4.0,0.0,3.0,1963.0,1.0


In [ ]:
df.index.name = "id"

column_map = {
                                            # Códigos territoriales INE
    "CPRO": "codigo_provincia",             # Código de provincia (INE). Cantabria = 39
    "CMUN": "codigo_municipio",             # Código de municipio (INE). Santa Cruz de Bezana = 073
    "TIPO_VIV": "tipo_vivienda",            # Tipo de vivienda: 1 = vivienda principal, 0 = no principal
    "TENEN_VIV": "regimen_tenencia",        # 2 = propiedad, 3 = alquiler, 4 = otras formas
    "RT_imp": "imputacion_RT",              # Indicador de imputación de renta: 0 = no imputado, 1 = imputado
    "SUPERF": "superficie_util",            # Superficie útil en m². Valor 1000 indica truncamiento (>= 1000 m²)
    "PLANTAS_sr": "plantas_sobre_rasante",  # Número de plantas sobre rasante. 10 indica top-code (>= 10 plantas)
    "PLANTAS_br": "plantas_bajo_rasante",   # Número de plantas bajo rasante. 2 indica top-code (>= 2 plantas)
    "TIPO_EDIF": "tipo_edificio",           # Tipo de edificio:
                                                # 1 = edificio residencial con 1 vivienda (unifamiliar)
                                                # 2 = edificio residencial con 2 viviendas (bifamiliar)
                                                # 3 = edificio residencial con ≥3 viviendas (plurifamiliar)
                                                # 4 = edificio no residencial con viviendas
    "ANOCO": "ano_construccion",            # Año de construcción. 1900 indica truncamiento (<= 1900)
    "TAM_HOG_5": "tamano_hogar"             # Número de personas en el hogar. 5 indica top-code (>= 5 personas)
}

df_rename = df.rename(columns=column_map).copy()

df_rename.head(10)


,codigo_provincia,codigo_municipio,tipo_vivienda,regimen_tenencia,imputacion_RT,superficie_util,plantas_sobre_rasante,plantas_bajo_rasante,tipo_edificio,ano_construccion,tamano_hogar
id,,,,,,,,,,,
1,3,99,1,2.0,1.0,84.0,1.0,0.0,1.0,1980.0,1.0
2,24,991,1,2.0,0.0,94.0,1.0,0.0,1.0,1931.0,3.0
3,32,991,0,NaN,NaN,104.0,1.0,0.0,1.0,1980.0,NaN
4,25,993,1,2.0,0.0,152.0,1.0,0.0,1.0,2005.0,3.0
5,30,30,1,3.0,1.0,69.0,4.0,0.0,3.0,1963.0,1.0
6,10,37,0,NaN,NaN,75.0,6.0,0.0,3.0,1969.0,NaN
7,7,40,1,2.0,0.0,36.0,2.0,0.0,3.0,2008.0,1.0
8,48,993,0,NaN,NaN,78.0,2.0,2.0,3.0,1997.0,NaN
9,25,993,1,2.0,0.0,203.0,1.0,0.0,1.0,1933.0,3.0
